<a href="https://colab.research.google.com/github/LizaMilovanova/Labs/blob/main/%D0%9B%D0%A0_9_%D0%9C%D1%96%D0%BB%D0%BE%D0%B2%D0%B0%D0%BD%D0%BE%D0%B2%D0%B0_%D0%A4%D0%86%D0%A2_4_8_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%A8%D0%B0%D0%B1%D0%BB%D0%BE%D0%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Мілованова Єлизавета
**Група:** ФІТ 4-8
**Дата:** 16.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:

In [5]:
# - кількість унікальних користувачів;
unique_users = ratings_df['user_id'].nunique()
print(f"Кількість унікальних користувачів: {unique_users}")

# - кількість унікальних фільмів;
unique_movies = ratings_df['item_id'].nunique()
print(f"Кількість унікальних фільмів: {unique_movies}")

# - мінімальне, максимальне, середнє значення rating;
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

print(f"Мінімальна оцінка: {min_rating}")
print(f"Максимальна оцінка: {max_rating}")
print(f"Середня оцінка: {mean_rating:.2f}")

# - пропущені значення (за наявності).
missing_values = ratings_df.isnull().sum()
print("\nПропущені значення у кожній колонці:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53

Пропущені значення у кожній колонці:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних MovieLens 100K містить інформацію від 943 унікальних користувачів, які оцінили 1682 унікальних фільми. Оцінки знаходяться в діапазоні від 1.0 до 5.0, при цьому середній рейтинг становить приблизно 3.53, що свідчить про схильність користувачів ставити переважно позитивні оцінки. Пропущені значення у базовому датасеті повністю відсутні, тому дані готові до подальшого використання у моделях SVD та KNNBasic

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:

In [ ]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:

In [7]:
reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
trainset = train_data.build_full_trainset() # Створюємо об'єкт Trainset для навчання

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f"Кількість оцінок у trainset (Surprise): {trainset.n_ratings}")
print(f"Кількість оцінок у testset (Surprise): {len(testset)}")

Кількість оцінок у trainset (Surprise): 80000
Кількість оцінок у testset (Surprise): 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [8]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для моделі SVD сітка параметрів охоплює ключові фактори, що впливають на точність та перенавчання. n_factors (50, 100) визначає розмірність прихованого простору характеристик користувачів та фільмів; більше факторів дозволяє вловити складніші залежності, але збільшує ризик перенавчання. lr_all (0.005, 0.010) — швидкість навчання для стохастичного градієнтного спуску, що контролює крок оновлення ваг. reg_all (0.02, 0.05) — параметр регуляризації, який штрафує за занадто великі значення векторів, запобігаючи перенавчанню.
Для моделі KNNBasic параметри зосереджені на алгоритмі пошуку сусідів. k (20, 40) задає максимальну кількість сусідів для прогнозування. min_k (1, 5) — мінімальна кількість спільних оцінок для врахування сусіда. Параметри sim_options визначають метрику подібності: name ('cosine' або 'pearson') порівнює косинусну відстань та коефіцієнт кореляції Пірсона, а user_based (True, False) дозволяє перевірити обидва підходи — колаборативну фільтрацію на основі користувачів (user-user) та на основі предметів (item-item). Такі значення обрані як стандартні стартові точки, які добре працюють на датасеті ml-100k

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:

In [9]:
grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)

In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:

In [10]:
grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(train_data)

In [ ]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:

In [11]:
print("=== Результати GridSearchCV для SVD ===")
print(f"Найкращий CV RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE: {grid_search_svd.best_score['mae']:.4f}")
print(f"Найкращі параметри (за RMSE): {grid_search_svd.best_params['rmse']}")

print("\n=== Результати GridSearchCV для KNNBasic ===")
print(f"Найкращий CV RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE: {grid_search_knn.best_score['mae']:.4f}")
print(f"Найкращі параметри (за RMSE): {grid_search_knn.best_params['rmse']}")

=== Результати GridSearchCV для SVD ===
Найкращий CV RMSE: 0.9449
Найкращий CV MAE: 0.7444
Найкращі параметри (за RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== Результати GridSearchCV для KNNBasic ===
Найкращий CV RMSE: 1.0269
Найкращий CV MAE: 0.8145
Найкращі параметри (за RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9449 | 0.7444 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0269 | 0.8145 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

За результатами перехресної перевірки (GridSearchCV) модель SVD показала кращу якість порівняно з KNNBasic. Її похибка є нижчою: RMSE становить 0.9449 проти 1.0269 у KNNBasic, а MAE — 0.7444 проти 0.8145.   

Для SVD найкращими параметрами виявилися 50 прихованих факторів (n_factors), швидкість навчання 0.01 (lr_all) та регуляризація 0.05 (reg_all). Для KNNBasic оптимальною стала конфігурація з 40 сусідів (k), мінімальним min_k=1, використанням косинусної міри подібності та підходом на основі користувачів (user_based: True). Загалом, алгоритм матричної факторизації SVD виявився точнішим для цього набору даних.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

In [12]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:

In [13]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [ ]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:

In [14]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

# Сортування за RMSE (від найменшої похибки до найбільшої)
results_sorted = results.sort_values(by='RMSE').reset_index(drop=True)

display(results_sorted)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки на незалежній тестовій вибірці не відрізняються суттєво від результатів кросвалідації (CV), що свідчить про хорошу узагальнюючу здатність обох моделей та відсутність перенавчання. Більше того, метрики на тестовій вибірці виявилися навіть дещо кращими за CV: для SVD RMSE знизився з 0.9449 до 0.9192[cite: 9], а для KNNBasic — з 1.0269 до 1.0120[cite: 9]. Модель SVD впевнено зберігає свою перевагу над KNNBasic як на етапі перехресної перевірки, так і на відкладених тестових даних

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

In [16]:
user_id = '196'

best_model = best_svd

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f"Користувача {user_id} немає у навчальній вибірці (trainset).")
    inner_uid = None

if inner_uid is not None:
    user_ratings = trainset.ur[inner_uid]
    rated_inner_iids = set([item_id for (item_id, rating) in user_ratings])

    all_inner_iids = set(trainset.all_items())

    unrated_inner_iids = all_inner_iids - rated_inner_iids

    unrated_raw_iids = [trainset.to_raw_iid(inner_iid) for inner_iid in unrated_inner_iids]

    predictions = []
    for raw_iid in unrated_raw_iids:
        pred = best_model.predict(uid=user_id, iid=raw_iid)
        predictions.append((pred.iid, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)

    top_10 = predictions[:10]

    top_10_df = pd.DataFrame(top_10, columns=['item_id', 'predicted_rating'])

    print(f"Топ-10 рекомендацій для користувача {user_id}:")
    display(top_10_df)

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

За результатами кросвалідації (CV) для подальшої роботи було обрано алгоритм матричної факторизації SVD, оскільки він продемонстрував нижчу похибку (CV RMSE = 0.9449), ніж алгоритм на основі сусідів KNNBasic (CV RMSE = 1.0269). Найкращими гіперпараметрами для SVD стали n_factors=50, lr_all=0.01 та reg_all=0.05, тоді як для KNNBasic оптимальною виявилася конфігурація з k=40, min_k=1, косинусною мірою та підходом user_based=True.
Оцінювання на незалежній тестовій вибірці підтвердило перевагу SVD: підсумкове значення Test RMSE склало 0.9192 (проти 1.0120 у KNNBasic), а Test MAE — 0.7234 (проти 0.7984 у KNNBasic). Значущої різниці між результатами CV та тестування не виявлено (тестові метрики навіть дещо покращилися), що свідчить про стабільність обох моделей та відсутність перенавчання. Головним обмеженням отриманих рекомендацій є проблема «холодного старту»: побудовані моделі SVD та KNNBasic не здатні формувати точні персоналізовані рекомендації для абсолютно нових користувачів чи щойно доданих фільмів, які ще не мають історії оцінок у навчальній вибірці.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**